[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/templates/b_25_dropout_pure.ipynb)

# 🟢 Easy: Dropout without Flax

*Core Ops & Layers*
Problem 17's dropout, with no module to hold the RNG.

### Signature
```python
class MyDropout:
    def __init__(self, p): ...
    def __call__(self, x, key, deterministic=False): ...
```

Same class as the `nnx` version except that `__call__` takes the **key**.
There is no `rngs=` in the constructor, because there is no stream to hold.

### Rules
- `deterministic=True` → return `x` unchanged.
- `self.p == 0.0` → return `x` unchanged, exactly. Both conditions, not one.
- Otherwise keep each element with probability `1 - p` and divide the
  survivors by `1 - p`, so the expectation is unchanged (inverted dropout).

`p` and `deterministic` decide a Python branch, so under `jit` they are
static.

### The key is the whole point
In problem 17 you wrote `self.rngs.dropout()` and a fresh mask appeared each
call, because the stream advanced itself. Here **nothing advances anything**:

```python
layer(x, key)      # same key
layer(x, key)      # SAME MASK — not a bug
```

JAX random functions are pure, so the same key gives the same mask. Two
training steps need two keys, and that is the caller's job:

```python
key, sub = jax.random.split(key)
h = layer(h, sub)
```

That is what the module was hiding — and why a JAX training loop threads a key
through its carry (see `b_19`).

### Why divide by 1-p
So that `E[out] == x`. Doing it at training time ("inverted dropout") is what
lets inference be a plain no-op instead of a rescale.

### Why this exists alongside problem 17
Same class, same constructor, same `deterministic` flag as the `nnx` version —
only the key changes hands. In problem 17 `nnx.Rngs` held it and advanced it
for you; here it is an argument to `__call__`, which is the whole lesson.

In [ ]:
# Colab setup (no-op when running locally).
# jax-judge is not published on PyPI, so the judge is installed from the
# repo itself. Regenerate with JAXCODE_REPO=you/YourFork to point this at
# your own fork:  JAXCODE_REPO=you/JAXCode make notebooks
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q flax optax')
    get_ipython().run_line_magic(
        'pip', 'install -q git+https://github.com/YOUR-GITHUB-USERNAME/JAXCode.git')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

import jax
import jax.numpy as jnp


class MyDropout:
    """Inverted dropout. The key is an argument, not state."""

    def __init__(self, p):
        pass  # Replace this

    def __call__(self, x, key, deterministic=False):
        pass  # Replace this

In [ ]:
# 🔍 Scratch cell — poke at your implementation
import jax
import jax.numpy as jnp

drop = MyDropout(0.5)
x = jnp.ones((8,))
key = jax.random.key(0)

print("p=0.5       ", drop(x, key))
print("same key    ", drop(x, key), "  <- identical, by design")
print("split key   ", drop(x, jax.random.split(key)[0]))
print("deterministic", drop(x, key, deterministic=True))

big = jnp.ones((100000,))
out = MyDropout(0.3)(big, key)
print(f"\nmean over 100k: {float(jnp.mean(out)):.4f}  (should be ~1.0)")
print(f"fraction zeroed: {float(jnp.mean(out == 0)):.4f}  (should be ~0.3)")

In [ ]:
# ✅ SUBMIT — run this cell to check your solution
from jax_judge import check, hint, solution, status

check("dropout_pure")

# hint("dropout_pure")      # stuck? nudge without the answer
# solution("dropout_pure")  # spoiler: the reference implementation
# status()                  # your dashboard across all problems